In [ ]:
import torch
from pyannote.audio import Pipeline
from dotenv import load_dotenv
import os
import datasets
import librosa
from IPython.display import Audio
import pandas as pd
import re
import numpy as np
import sqlalchemy
from sqlalchemy import create_engine, text
import datetime


In [ ]:
device="mps" if torch.backends.mps.is_available() else "cpu"
device

In [ ]:
import whisper
model=whisper.load_model("base",device=device)

In [ ]:
load_dotenv()
hf_token=os.getenv("HF_TOKEN")

In [ ]:
# pipeline = Pipeline.from_pretrained(
#     "pyannote/speaker-diarization-3.1",
#     token=hf_token
# )

# pipeline.to(torch.device("mps"))

# diarization=pipeline("recording.m4a")



In [ ]:
# for turn, _, speaker in diarization.speaker_diarization.itertracks(yield_label=True):
#     print(f"Speaker {speaker}: {turn.start:.1f}s --> {turn.end:.1f}s")

In [ ]:
# from datasets import load_dataset
# ds = load_dataset("UniDataPro/call-center-audio")

In [ ]:
for sample in ds['train']:
    audio=sample['audio']
    decoded=audio.metadata
    print(decoded.duration_seconds/60,decoded.sample_rate,decoded.sample_format)

In [ ]:
audio=ds['train'][0]['audio']

In [ ]:
row1=audio.get_all_samples().data[1]

In [ ]:
row1

In [ ]:
trial_result=model.transcribe(row1,fp16=False)

In [ ]:
b=Audio(row1,rate=16000)

In [ ]:
trial_result

In [ ]:
transcript=trial_result["text"]

In [ ]:
transcript

In [ ]:
def clean_text(sentence):
    repetition_pattern = re.compile(r'\b([A-Za-z]+)(?:\s+\1\b)+', re.IGNORECASE)
    no_repetition=repetition_pattern.sub(r'\1',sentence)
    return no_repetition

In [ ]:
clean_text(transcript)

In [ ]:
ds2 = load_dataset("AIxBlock/English-role-playing-call-center-convers-different-moods")

In [ ]:
ds2['train']

In [ ]:
Audio(ds2['train'][9]['audio'].get_all_samples().data,rate=48000)

In [ ]:
Audio(ds2['train'][10]['audio'].get_all_samples().data,rate=48000)
ds2

In [ ]:

for sample in ds2['train']:
    audio=sample['audio']
    decoded=audio.metadata
    print(decoded.duration_seconds/60,decoded.sample_rate,decoded.sample_format)


Seeing the meta data above, there looks to be two problems:-
1. The assumption that the data is in pairs looks false. For eg row 5 is standalone
2. Last few rows have different sampling rate.
So what i have decided is first transcript all and then do some analysis on the data and give a label to each row whether its customer care or customer.

Problem- There are some pairs one after the another which have call centre conversation followed by customer conversation.
And there are some which have only one. I dont know whose data that is whether call centre or customer.
So when i run the voices through transcription model, i will get the transcription. But i need to know if it is customer/call-centre data and whether it has a pairing data of the other side. 
How do i do that?

What i am going to do right now is look at the call duration of the next row. If its almost same, then i am going to treat the current and the next as a pair.

In [ ]:
sample=ds2['train'][0]
audio=sample['audio']
audio_array=audio.get_all_samples().data
transcript=model.transcribe(audio_array[0],fp16=False)

In [ ]:
transcript

In [ ]:
engine = create_engine('sqlite:///callcentre.db')
with engine.connect() as conn:
    for sample in ds2['train']:
        audio=sample['audio']
        audio_array=audio.get_all_samples().data[0]
        transcript=model.transcribe(audio_array,fp16=False)
        metadata=audio.metadata
        conn.execute(text('''
            INSERT INTO conversations (pair_index, timestamp, transcript, speaker,call_duration, is_paired)
            VALUES (:pair_index, :timestamp, :transcript, :speaker,:call_duration, :is_paired)
        '''), {
            'pair_index': 0,
            'timestamp': datetime.datetime.now(),
            'transcript': transcript['text'],
            'speaker': 'agent',
            'call_duration': metadata.duration/60,
            'is_paired': True
        })
        conn.commit()

